# Введение в детекцию, сегментацию и их метрики

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Различать классификацию, детекцию и сегментацию и вычислять IoU.

## Где находится ответ

Классификация отвечает, что изображено на картинке. Детекция указывает объекты рамками и метками. Семантическая сегментация присваивает класс каждому пикселю. Поэтому одна accuracy картинки не оценивает правильность координат и масок.

Рамку в этом уроке задаём координатами (лево,верх,право,низ), ширина равна право−лево. Границы интерпретируем как координаты непрерывной области, без дополнительной единицы. В других форматах соглашение может отличаться, поэтому его записывают явно.

## Разобранный пример: перекрытие

Правильная рамка A=[0,0,2,2], прогноз B=[1,0,3,2]. Площади обеих 4, пересечение имеет ширину 1 и высоту 2, площадь 2. Объединение 4+4−2=6. **IoU**, пересечение, делённое на объединение, равно 1/3.

Для масок IoU считает общие положительные пиксели, делённые на все пиксели хотя бы одной маски. Если обе маски пусты, знаменатель ноль: нужно заранее выбрать соглашение, а не получить случайное деление.

## Метрики требуют сопоставления

В детекции несколько прогнозов нужно сопоставить с правильными объектами по классу и порогу IoU. Две рамки вокруг одного объекта не являются двумя верными обнаружениями. Precision и recall зависят от порога уверенности; AP суммирует качество по ранжированному списку при определённом протоколе. Это обзор смысла метрик, не самостоятельная реализация полного стандарта оценки.

В ноутбуке вручную вычисляем IoU пары рамок и масок. Модели и изображения не скачиваем. Такая практика проверяет геометрию ответа и граничные случаи до обучения тяжёлого детектора.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

В `box_iou` сначала проверяем число, конечность и порядок координат, затем находим ширину и высоту пересечения отдельно. `math.isfinite` уже использовался при проверке файлов. Рамка нулевой площади допустима; две пустые области здесь отклоняются из-за нулевого объединения. Для масок знак `&` задаёт пересечение, `|` — объединение, а сумма логических значений считает пиксели.

In [ ]:
import math
import numpy as np
def box_iou(a, b):
    for box in [a, b]:
        if len(box) != 4 or not all([math.isfinite(v) for v in box]):
            raise ValueError("Рамка должна содержать четыре конечные координаты")
        if box[2] < box[0] or box[3] < box[1]:
            raise ValueError("Правая и нижняя границы не должны предшествовать левой и верхней")
    width = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    height = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    intersection = width * height
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - intersection
    if union == 0:
        raise ValueError("Обе области пусты")
    return intersection / union
assert np.isclose(box_iou([0,0,2,2], [1,0,3,2]), 1/3)
a = np.array([[1, 1], [0, 0]], dtype=bool)
b = np.array([[0, 1], [0, 1]], dtype=bool)
mask_union = (a | b).sum()
if mask_union == 0:
    raise ValueError("IoU двух пустых масок требует отдельного соглашения")
assert np.isclose((a & b).sum() / mask_union, 1/3)
print("IoU рамок и масок: 1/3")


## Типичные ошибки

Проверяйте формат координат и пустые области. Не смешивайте разные правила включения границ рамки.

## Задачи

### 1. Базовая

Чем сегментация отличается от классификации картинки?

<details>
<summary>Подсказка</summary>

Ответ привязан к каждому месту.

</details>



### 2. Базовая

В каких пределах IoU непустых областей?

<details>
<summary>Подсказка</summary>

Пересечение не больше объединения.

</details>



### 3. Прикладная

Площади 4 и 4, пересечение 2. Найдите IoU.

<details>
<summary>Подсказка</summary>

Объединение вычитает повтор.

</details>



### 4. Прикладная

Рамки не пересекаются. Каков IoU при положительных площадях?

<details>
<summary>Подсказка</summary>

Числитель ноль.

</details>



### 5. Олимпиадная

Почему два прогноза одного объекта не должны считаться двумя TP?

<details>
<summary>Подсказка</summary>

Иначе можно улучшать счёт копированием рамки.

</details>



### 6. Олимпиадная
На изображении 100 пикселей, 5 принадлежат объекту. Модель везде предсказала фон. Найдите accuracy и IoU объекта. Может ли высокая accuracy скрыть полный провал сегментации?

<details>
<summary>Подсказка</summary>

Для объекта пересечение пусто, а объединение непусто.

</details>



## Итоги и самопроверка

Вручную нарисуйте две рамки и восстановите IoU. Что нужно уточнить перед оценкой набора детекций?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Она выдаёт метки пикселей, а не одну метку всей картинки.

### Задача 2

От 0 до 1.

### Задача 3

2/(4+4−2)=1/3.

### Задача 4

0.

### Задача 5

Правильный объект сопоставляют ограниченному числу прогнозов по протоколу, дубли становятся лишними обнаружениями.

### Задача 6

Верно размечены 95 фоновых пикселей, accuracy=95/100=0.95. Для объекта пересечение равно 0, объединение содержит 5 пикселей, IoU=0/5=0. Модель полностью пропустила объект несмотря на 95% accuracy; нужна оценка целевой области.